# TRL Benchmarking on OGBench

This notebook runs the Transitive RL pipeline on the full battery of OGBench tasks like in the original paper, as well as running all of the other algorithms found in OGBench. It is organized as:

1. **Environment setup** — GPU runtime, installs, path config
2. **Project Root setup** — make sure that your files can be found correctly
3. **Sanity checks** — confirm JAX sees the GPU, imports resolve, TRL agent initializes
4. **Smoke test** — 500 steps with tiny batch; assert the pipeline doesn't crash and losses behave
5. **Full training run** — 1M steps, eval every 100K, checkpoint to Drive

**Before you run anything:** Runtime → Change runtime type → **A100 GPU**. If you don't do this, everything will work, but training will be ~10× slower and you will be sad.

The only thing that need to be changed is the filepath in section 2 to match your own filepath.

## 1. Environment setup

In [ ]:
# Confirm we have a GPU before installing anything heavy.
!nvidia-smi

Thu May  7 01:09:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Install repo requirements. Pinned to minimize Colab-specific breakage.
# JAX CUDA wheels need to match the driver; Colab's current default works with jax[cuda12].
# If JAX-CUDA detection fails later, the most common fix is reinstalling the matching wheel.
!pip install -q \
    "jax[cuda12]>=0.4.26" \
    "flax>=0.8.4" \
    "distrax>=0.1.5" \
    ml_collections \
    matplotlib \
    moviepy \
    wandb \
    ogbench \
    imageio \
    tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.3/313.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 99.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [ ]:
# Mount Drive to persist dataset downloads and checkpoints across sessions.
# OGBench datasets are ~1GB and redownloading every session is painful.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ['MUJOCO_GL'] = 'egl'

Mounted at /content/drive


In [ ]:
# log into wandb
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ljcho2005 (ljcho2005-princeton-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Upload & unpack the repo

Add your own root filepath (aka everything before cos435-final-project/...) so that the notebook can find your files correctly

 (NOTE: please comment out other people's filepaths when you run your own version)

In [ ]:
# Veronica's filepath
root = '/content/drive/MyDrive'

In [ ]:
# Luke's filepath
root = "/content/drive/MyDrive/Year 3/Sem 2/COS435"

# his other filepath
# root = "/content/drive/MyDrive/Classes"



In [ ]:
# Tate's filepath
# root =

In [ ]:
# Cheryl's filepath
# root =

In [ ]:
# Maya's filepath
root = '/content/drive/MyDrive'

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(root)

## 3. Sanity checks

Before burning GPU time, confirm the obvious things work: JAX sees the GPU

In [ ]:
# Check: JAX sees the GPU
import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())
assert any('gpu' in str(d).lower() or 'cuda' in str(d).lower() for d in jax.devices()), (
    'No GPU found. Go to Runtime -> Change runtime type -> A100 GPU.'
)

JAX version: 0.7.2
Devices: [CudaDevice(id=0)]


## 4. Smoke test
This is a short smoke test using the test_hyperparameters.py file (which uses 500 step runs using `pointmaze-medium-navigate-oraclerep-v0`). If something goes wrong, it probably should be fixed here before running EVERYTHING else. I have no idea how long this should take on A100s, but hopefully not long.

In [ ]:
import os
import re
import hashlib
import subprocess
from datetime import datetime
from tqdm import tqdm

def run_experiments(commands, output_dir):
    output_dir = os.path.abspath(output_dir)

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(output_dir, f"run_{run_id}")
    os.makedirs(run_dir, exist_ok=True)

    log_path = os.path.join(run_dir, "logs.txt")

    print(f"Starting run: {run_dir}")
    print("CWD before:", os.getcwd())

    os.chdir(PROJECT_ROOT / "cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls")

    print("CWD after:", os.getcwd())
    pbar = tqdm(total=len(commands), desc="Running experiments", dynamic_ncols=True)
    for i, cmd in enumerate(commands):
        print(f"\n[{i+1}/{len(commands)}] Running: {cmd}")

        try:
            result = subprocess.run(
                cmd,
                shell=True,
                capture_output=True,
                text=True,
                env=os.environ.copy()
            )
        except Exception as e:
            print(f"EXCEPTION: {e}")
            continue

        with open(log_path, "a") as f:
            f.write("\n--- CMD ---\n")
            f.write(cmd + "\n")
            f.write("\n--- STDOUT ---\n")
            f.write(result.stdout or "")
            f.write("\n--- STDERR ---\n")
            f.write(result.stderr or "")
            f.write("\n--- RETURN CODE ---\n")
            f.write(str(result.returncode) + "\n")

        print("Return code:", result.returncode)

        if result.returncode != 0:
            print("FAILED — check logs.txt")
        pbar.update(1)

    pbar.close()

In [ ]:
# parse the commands
hp_path = PROJECT_ROOT / "cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls/test_hyperparams.sh"

with open(hp_path, "r") as f:
    hp_content = f.read()

test_commands = []
for line in hp_content.split("\n"):
    line = line.strip()

    if not line or line.startswith("#"):
        continue

    line = line.split("#")[0].strip()

    if line.startswith("python"):
        test_commands.append(line)

print(test_commands)

print(f"Found {len(test_commands)} commands in test_hyperparams.sh\n")

['python main.py --env_name=puzzle-3x3-play-v0 --agent=agents/trl.py --run_group=Test_3 --run_group=Test --train_steps=500 --eval_interval=500 --agent.actor_hidden_dims="(512, 512, 512)" --agent.value_hidden_dims="(512, 512, 512)" --agent.actor_geom_sample=False --agent.actor_p_trajgoal=1.0 --agent.actor_p_randomgoal=0.0 --agent.value_geom_sample=True --agent.discount=0.99 --agent.use_oracle_distillation=True --agent.expectile=0.7 --agent.distance_weight_lambda=0.7 --agent.policy_extraction=ddpgbc --agent.alpha=2']
Found 1 commands in test_hyperparams.sh



In [ ]:
# smoke test
run_experiments(test_commands, PROJECT_ROOT / "cos435-final-project/test-runs/checkpoints")

Starting run: /content/drive/MyDrive/cos435-final-project/test-runs/checkpoints/run_20260507_011150
CWD before: /content/drive/MyDrive/cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls
CWD after: /content/drive/MyDrive/cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls



Running experiments:   0%|          | 0/1 [00:00<?, ?it/s]


[1/1] Running: python main.py --env_name=puzzle-3x3-play-v0 --agent=agents/trl.py --run_group=Test_3 --run_group=Test --train_steps=500 --eval_interval=500 --agent.actor_hidden_dims="(512, 512, 512)" --agent.value_hidden_dims="(512, 512, 512)" --agent.actor_geom_sample=False --agent.actor_p_trajgoal=1.0 --agent.actor_p_randomgoal=0.0 --agent.value_geom_sample=True --agent.discount=0.99 --agent.use_oracle_distillation=True --agent.expectile=0.7 --agent.distance_weight_lambda=0.7 --agent.policy_extraction=ddpgbc --agent.alpha=2



Running experiments: 100%|██████████| 1/1 [02:57<00:00, 177.35s/it]

Return code: 0


## 5. Full Runs

This is where the main evaluation will happen. You dont need to modify anything.

In [ ]:
# parse the commands for the original 3 long-horizon tasks
hp_path = PROJECT_ROOT / "cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls/eval_hyperparameters.sh"

with open(hp_path, "r") as f:
    hp_content = f.read()

commands = []
for line in hp_content.split("\n"):
    line = line.strip()

    if not line or line.startswith("#"):
        continue

    line = line.split("#")[0].strip()

    if line.startswith("python"):
        commands.append(line)

print(f"Found {len(commands)} commands in eval_hyperparams.sh\n")

for command in commands:
  print(command)

Found 7 commands in eval_hyperparams.sh

python main.py --env_name=antmaze-large-stitch-v0 --agent=agents/trl.py --train_steps=1000000 --agent.actor_hidden_dims="(512, 512, 512)" --agent.value_hidden_dims="(512, 512, 512)" --agent.actor_geom_sample=False --agent.actor_p_trajgoal=1.0 --agent.actor_p_randomgoal=0.0 --agent.value_geom_sample=True --agent.discount=0.99 --agent.use_oracle_distillation=False --agent.expectile=0.7 --agent.distance_weight_lambda=0.0 --agent.policy_extraction=ddpgbc --agent.alpha=0.7
python main.py --env_name=pointmaze-teleport-navigate-v0 --agent=agents/trl.py --train_steps=1000000 --agent.actor_hidden_dims="(512, 512, 512)" --agent.value_hidden_dims="(512, 512, 512)" --agent.actor_geom_sample=False --agent.actor_p_trajgoal=1.0 --agent.actor_p_randomgoal=0.0 --agent.value_geom_sample=True --agent.discount=0.99 --agent.use_oracle_distillation=False --agent.expectile=0.7 --agent.distance_weight_lambda=0.7 --agent.policy_extraction=ddpgbc --agent.alpha=10
python 

In [ ]:
# run the experiment
run_experiments(commands, PROJECT_ROOT / "cos435-final-project/test-runs/checkpoints")

Starting run: /content/drive/MyDrive/cos435-final-project/test-runs/checkpoints/run_20260507_065228
CWD before: /content/drive/MyDrive/cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls
CWD after: /content/drive/MyDrive/cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls


Running experiments:   0%|          | 0/4 [00:00<?, ?it/s]


[1/4] Running: python main.py --env_name=puzzle-4x6-play-v0 --agent=agents/trl.py --train_steps=1000000 --agent.actor_hidden_dims="(1024, 1024, 1024, 1024)" --agent.value_hidden_dims="(1024, 1024, 1024, 1024)" --agent.discount=0.999 --agent.actor_geom_sample=True --agent.actor_p_trajgoal=0.5 --agent.actor_p_randomgoal=0.5 --agent.value_geom_sample=False --agent.value_p_trajgoal=1.0 --agent.value_p_randomgoal=0.0 --agent.use_oracle_distillation=False --agent.expectile=0.7 --agent.distance_weight_lambda=0 --agent.policy_extraction=rejection


Running experiments:  25%|██▌       | 1/4 [2:44:49<8:14:29, 9889.86s/it]

Return code: 0

[2/4] Running: python main.py --env_name=puzzle-4x4-play-oraclerep-v0 --agent=agents/trl.py --train_steps=1000000 --agent.actor_hidden_dims="(512, 512, 512)" --agent.value_hidden_dims="(512, 512, 512)" --agent.actor_geom_sample=False --agent.actor_p_trajgoal=1.0 --agent.actor_p_randomgoal=0.0 --agent.value_geom_sample=True --agent.discount=0.99 --agent.use_oracle_distillation=True --agent.expectile=0.7 --agent.distance_weight_lambda=2.0 --agent.policy_extraction=rejection


...

Return code: 0

[3/4] Running: python main.py --env_name=humanoidmaze-medium-navigate-oraclerep-v0 --agent=agents/trl.py --train_steps=1000000 --agent.actor_hidden_dims="(512, 512, 512)" --agent.value_hidden_dims="(512, 512, 512)" --agent.actor_geom_sample=False --agent.actor_p_trajgoal=1.0 --agent.actor_p_randomgoal=0.0 --agent.value_geom_sample=True --agent.discount=0.995 --agent.use_oracle_distillation=True --agent.expectile=0.7 --agent.distance_weight_lambda=0.0 --agent.policy_extraction=rejection


Running experiments:  75%|███████▌  | 3/4 [6:00:53<1:54:46, 6886.49s/it]

Return code: 0

[4/4] Running: python main.py --env_name=humanoidmaze-large-navigate-oraclerep-v0 --agent=agents/trl.py --train_steps=1000000 --agent.actor_hidden_dims="(512, 512, 512)" --agent.value_hidden_dims="(512, 512, 512)" --agent.actor_geom_sample=False --agent.actor_p_trajgoal=1.0 --agent.actor_p_randomgoal=0.0 --agent.value_geom_sample=True --agent.discount=0.995 --agent.use_oracle_distillation=True --agent.expectile=0.7 --agent.distance_weight_lambda=0.1 --agent.policy_extraction=rejection


Running experiments: 100%|██████████| 4/4 [8:01:16<00:00, 7219.20s/it]

Return code: 0
